In [0]:
%run ../functions/functions

In [0]:
# Nome do banco de dados onde a tabela será criada ou utilizada
database_name = "dimensao"

# Nome da tabela de calendário
table_name = "dm_calendario"

# Caminho alvo no formato <banco>.<tabela>
target_path = f"{database_name}.{table_name}"

# Nome da chave primária da tabela de calendário
pk = "PK_CALENDARIO"

In [0]:
query = """
/*
  Gera uma query SQL para criar uma tabela de dimensão calendário (dm_calendario)
  cobrindo datas de 01/01/2000 a 31/12/2050, com diversas colunas úteis para análises temporais,
  incluindo flags de início/fim de mês/ano, nomes de meses/dias em português, e indicadores de fim de semana/dia útil.
*/

WITH calendario AS (

  /* Gera uma lista de datas de 2000-01-01 até 2050-12-31, uma linha por dia */
  SELECT explode(
    sequence(
      to_date('2000-01-01'),
      to_date('2050-12-31'),
      interval 1 day
    )
  ) as data

)

SELECT
  data, -- Data no formato date

  -- Chave numérica (Data Warehouse)
  CAST(date_format(data, 'yyyyMMdd') AS INT) as sk_data,

  -- Ano da data
  year(data) as ano,

  -- Semestre (1 para Jan-Jun, 2 para Jul-Dez)
  CASE
    WHEN month(data) <= 6 THEN 1
    ELSE 2
  END as semestre,

  -- Trimestre do ano (1 a 4)
  quarter(data) as trimestre,

  -- Número do mês (1 a 12)
  month(data) as mes,

  -- Nome do mês em português
  CASE month(data)
    WHEN 1 THEN 'Janeiro'
    WHEN 2 THEN 'Fevereiro'
    WHEN 3 THEN 'Março'
    WHEN 4 THEN 'Abril'
    WHEN 5 THEN 'Maio'
    WHEN 6 THEN 'Junho'
    WHEN 7 THEN 'Julho'
    WHEN 8 THEN 'Agosto'
    WHEN 9 THEN 'Setembro'
    WHEN 10 THEN 'Outubro'
    WHEN 11 THEN 'Novembro'
    WHEN 12 THEN 'Dezembro'
  END as nome_mes,

  -- Nome abreviado do mês em português
  CASE month(data)
    WHEN 1 THEN 'Jan'
    WHEN 2 THEN 'Fev'
    WHEN 3 THEN 'Mar'
    WHEN 4 THEN 'Abr'
    WHEN 5 THEN 'Mai'
    WHEN 6 THEN 'Jun'
    WHEN 7 THEN 'Jul'
    WHEN 8 THEN 'Ago'
    WHEN 9 THEN 'Set'
    WHEN 10 THEN 'Out'
    WHEN 11 THEN 'Nov'
    WHEN 12 THEN 'Dez'
  END as nome_mes_abrev,

  -- Ano e mês no formato yyyy-M
  date_format(data, 'yyyy-M') as ano_mes,

  -- Dia do mês (1 a 31)
  day(data) as dia_mes,

  -- Dia do ano (1 a 366)
  dayofyear(data) as dia_ano,

  -- Semana do ano (1 a 53)
  weekofyear(data) as semana_ano,

  -- Dia da semana numérico (1=domingo, 7=sábado)
  dayofweek(data) as dia_semana_num,

  -- Nome do dia da semana em português
  CASE dayofweek(data)
    WHEN 1 THEN 'Domingo'
    WHEN 2 THEN 'Segunda-feira'
    WHEN 3 THEN 'Terça-feira'
    WHEN 4 THEN 'Quarta-feira'
    WHEN 5 THEN 'Quinta-feira'
    WHEN 6 THEN 'Sexta-feira'
    WHEN 7 THEN 'Sábado'
  END as nome_dia,

  -- Flag indicando se é fim de semana (domingo ou sábado)
  CASE
    WHEN dayofweek(data) IN (1,7) THEN true
    ELSE false
  END as fim_de_semana,

  -- Flag indicando se é dia útil (segunda a sexta)
  CASE
    WHEN dayofweek(data) IN (2,3,4,5,6) THEN true
    ELSE false
  END as dia_util,

  -- Primeiro dia do mês
  trunc(data, 'MM') as inicio_mes,

  -- Último dia do mês
  last_day(data) as fim_mes,

  -- Primeiro dia do ano
  trunc(data, 'YYYY') as inicio_ano,

  -- Último dia do ano
  add_months(trunc(data, 'YYYY'), 12) - interval 1 day as fim_ano,

  -- Flag indicando se é o primeiro dia do mês
  CASE WHEN data = trunc(data,'MM') THEN true ELSE false END as eh_inicio_mes,
  -- Flag indicando se é o último dia do mês
  CASE WHEN data = last_day(data) THEN true ELSE false END as eh_fim_mes,

  -- Flag indicando se é o primeiro dia do ano
  CASE WHEN data = trunc(data,'YYYY') THEN true ELSE false END as eh_inicio_ano,
  -- Flag indicando se é o último dia do ano
  CASE WHEN data = add_months(trunc(data,'YYYY'),12) - interval 1 day THEN true ELSE false END as eh_fim_ano

FROM calendario;

"""

In [0]:
# Executa a query SQL definida anteriormente e armazena o resultado em um DataFrame Spark
df_final = spark.sql(query)

In [0]:
# Cria o banco de dados se ele não existir
spark.sql(f"CREATE DATABASE IF NOT EXISTS database_name")

In [0]:
# Salva o DataFrame df_final como uma tabela Hive no caminho especificado (target_path),
# utilizando a coluna pk como chave primária.
save_hive_table(df_final, target_path, pk)